# 'Skeleton' Utils

In [3]:
#import pckgs
import cv2
import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from natsort import natsorted
import re

### Delete Files with a specific string in directory 
Always run it with #os.remove commented first!

In [ ]:
main_path='/Volumes/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/'

for roots, dirs, files in natsorted(os.walk(main_path)):
    for single_dir in dirs:
        for single_file in natsorted(files):
#            print((os.path.join(roots,single_dir,single_file)))
            #print(files)
            if 'compressed' in single_file and 'ome.tif' not in single_file:
                print(os.path.join(roots, single_file))
                #os.remove(os.path.join(roots, single_file))
                #break

### Tiff Writer example

In [ ]:
with tiff.TiffWriter(output_filename, bigtiff=True) as tif_writer:
    for img in video:
        tif_writer.save(img, photometric='minisblack', description=omexmlMetadataString)

## CSV Writer example

In [6]:
#file=open("second_derivative.csv", K)
import csv
csvfile=open('persons.csv','w', newline='')
csv_writer=csv.writer(csvfile)
csv_writer.writerow('hello')

11

In [9]:
import csv
import numpy as np
persons=[('Lata',22,45),('Anil',21,56),('John',20,60)]
K=np.ones(10)
csvfile=open('personsK.csv','w', newline='')
csv_writer=csv.writer(csvfile)
for person in persons:
    csv_writer.writerow(K)
csvfile.close()

### OPENCV reader and writer example

In [1]:
#Load input .avi and create output .avi
#video reader
file_path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/all_avi/2020-06-05_16-46-41_worm1-channel-0-bigtiff.avi'
video_cap = cv2.VideoCapture(file_path)
#video writer
fps=167
frame_width = int(video_cap.get(3))
frame_height = int(video_cap.get(4))
output_filepath='/groups/zimmer/Ulises/code/deeplabcut_projects/outpy7.avi'
video_out = cv2.VideoWriter(output_filepath,cv2.VideoWriter_fourcc('M','J','P','G'), fps, (frame_width,frame_height))
k=0
while(video_cap.isOpened()):
    ret,frame=video_cap.read()
    if ret == True:
        output=frame.copy()

        video_out.write(output)
        k=k+1
    if k>2000:
        video_cap.release()
        #video_cap.release()
        video_out.release()
        cv2.destroyAllWindows()

NameError: name 'cv2' is not defined

### ometiff2bigtiff
The function reads all the ometiff files in a directory and saves them in a big tiff file with the name of the directory + bigtiff.btf
It uses Sequential reading as Lukas showed me.

The code below is based on a notebook with the same name we wrote which is stored in /shared_projects/notebooks/lukas_notebooks

We didnt manage to save as .avi the big tiff file and decided to do it with fiji instead, which seems to work.

In [4]:
#function to list all ome.tiff in a directory and make them one bigtiff

#somehow it gives an error for the last ome tiff, but the movie is fine.
def ometiff2bigtiff(path):
    if path.endswith('/'):
        output_filename=path+re.split('/',path)[-2]+'bigtiff.btf'
    else:
        output_filename=path+'/'+re.split('/',path)[-1]+'bigtiff.btf'
    with tiff.TiffWriter(output_filename, bigtiff=True) as output_tif:
        for file in natsorted(os.listdir(path)):
            #print(os.path.join(path,file))
            if file.endswith('ome.tif') and 'bg' not in file:
                print(os.path.join(path,file))
                with tiff.TiffFile(os.path.join(path,file), multifile=False) as tif:
                    #print('entered writing')
                    hyperstack = tif.asarray()
                    omexmlMetadataString = tif.ome_metadata #IF YOU RUN THIS LINE IT GIVES ERRORS!
                    #print('writing...')
                    output_tif.save(hyperstack, photometric='minisblack', description=omexmlMetadataString)

In [1]:
root='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/20191113_cal01/control/worm1/rec1/data/bh/ome/'

In [5]:
ometiff2bigtiff(root)

/groups/zimmer/Ulises/wbfm/chemotaxis_assay/20191113_cal01/control/worm1/rec1/data/bh/ome/MMStack.ome.tif


UnicodeEncodeError: 'ascii' codec can't encode character '\xb5' in position 540: ordinal not in range(128)

In [ ]:
#try single ome.tif.
path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/dataset_20200607/'
file='2020-06-05_16-46-41_worm1-channel-0-_MMStack_20.ome.tif'
with tiff.TiffFile(os.path.join(path,file), multifile=False) as tif:
    hyperstack = tif.asarray()
#    omexmlMetadataString = tif.ome_metadata
    tiff.imsave((path+'test_new20.tif'), hyperstack)#, description=omexmlMetadataString)

In [ ]:
main_path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/dataset_20200605/'

for roots, dirs, files in natsorted(os.walk(main_path)):
    print(dirs)
    for single_dir in natsorted(dirs):
        if 'worm' in single_dir and 'bg' not in single_dir:
            print('the directory is:')
            print(os.path.join(roots,single_dir)+'\n')
            ometiff2bigtiff(os.path.join(roots,single_dir))

### Compress ometiff file
Compress it to smaller (in dimensions) tiff file

In [ ]:
#define compress_ometif function
def compress_ometif(file):
    #print('reading ome tif file: '+file+'\n... might take a while')
    retval, mats=cv2.imreadmulti(file)
    #print('reading complete')
    ometif=np.asarray(mats)
    new_img=np.full((ometif.shape[0],200,200), 0, dtype='uint8')
    #print('entering the loop')
    for k,img in enumerate(ometif):
        resized = cv2.resize(img, (200,200), interpolation = cv2.INTER_AREA)
        new_img[k]=resized
    return new_img
    #tiff.imsave(filename,new_img, bigsize=True) 

In [ ]:
#generate avi files for the ome.tif files in the directory
main_path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/dataset_20200701/'

#for every root directory
for roots, dirs, files in os.walk(main_path):
    #for every file in roots
    for file in natsorted(files):
        if file.endswith('ome.tif') and 'bg' not in file:
            #print(os.path.join(roots,file))
            new_img=compress_ometif(os.path.join(roots,file))
            tiff.imsave(os.path.join(roots, re.split('.ome.tif',file)[0])+ '_compressed.tiff', new_img, bigsize=True)
print('end')
        

In [ ]:
#TEST merge all files in roots
print('!!!before running this I changed the Name of ...MMStack_compressed to ...MMStack_0_compressed in NameChanger\n\n\n')
main_path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/dataset_20200701/2020-07-01_18-36-25_control_worm6-channel-0-/'
for roots, dirs, files in natsorted(os.walk(main_path)):
    if 'worm' in roots and 'bg' not in roots:
        output_video=np.full((0,200,200), 0, dtype='uint8')
        #for every file in roots
        for file in natsorted(files):
            if file.endswith('.avi'):
                print(os.path.join(roots,file))
                mats=tiff.imread(os.path.join(roots,file))
                avi_file=np.asarray(mats)
                output_video=np.concatenate([output_video,avi_file],0)
                #print(output_video.shape)
        print(os.path.join(roots,re.split('/',roots)[-1])+'_ALL_compressed.tiff')
        tiff.imsave(os.path.join(roots,re.split('/',roots)[-1])+'_ALL_compressed.tiff', output_video, bigsize=True)
print('end')

In [ ]:
path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/dataset_20200605/\
2020-06-05_16-46-41_worm1-channel-0-/'

output_video=np.full((0,200,200), 0, dtype='uint8')

for file in natsorted(os.listdir(path)):
    print(file)
    if file.endswith('ome.tif'):
        print(file)
        new_img=compress_ometif(path,file)
        output_video=np.concatenate([output_video,new_img],0)
tiff.imsave(path+'ourput.tiff',output_video, bigsize=True)
print('end')

## Draw BodyParts


In [12]:
##Make video binary
#read and write tiff video
import tifffile as tiff
import cv2

output_filename='/groups/zimmer/Ulises/code/skeleton_outputs/new3/img.tiff'

input_filename='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/all_btf/2020-06-05_16-46-41_worm1-channel-0-bigtiff.btf'

bg_img= tiff.imread('/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/dataset_20200605/2020-06-05_16-46-41_worm1-channel-0-/2020-06-05_16-46-41_worm1-channel-0-copied_bg/2020-06-09_10-21-16_control_worm1_bg-channel-0-_MMStack_Median_BG.tiff')

sec_der=[]
knots=100
with tiff.TiffWriter(output_filename, bigtiff=True) as tif_writer:
    with tiff.TiffFile(input_filename, multifile=False) as tif:
        for i, page in enumerate(tif.pages):
            img=page.asarray()
            img=cv2.bitwise_not(img)

            img=cv2.subtract(img,bg_img)

            #median Blur
            img[:] = cv2.medianBlur(img,5)
            #apply threshold
            ret, new_img = cv2.threshold(img,11,255,cv2.THRESH_BINARY)
            #find contours
            contours = cv2.findContours(new_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            contours = contours[0] if len(contours) == 2 else contours[1]
            #list areas of contours, find MAX, draw contours from MAX area
            areas=[]
            for j in range(0, len(contours)):
                areas.append(cv2.contourArea(contours[j]))
            worm_contour=np.where(areas==np.asarray(areas).max())
            worm_contour=np.asarray(worm_contour)

            img_contours = np.zeros(img.shape)
            img[:]=cv2.drawContours(img_contours,contours, worm_contour, 255, -1)
            tif_writer.save(img)


KeyboardInterrupt: 

In [ ]:
#to plot head and tail
#n_frames=500
#new_video=video[0:n_frames].copy()
new_video=video.copy()

for k, img in enumerate(new_video):
    cv2.circle(img, (int(head_x[k]), int(head_y[k])), 20, (255,0,0), 2)
    cv2.circle(img, (int(tail_x[k]), int(tail_y[k])), 5, (255,0,0), 2)
    
#file variable contains the file name from few cells above
tiff.imsave(output_path+'head_tail_annotation'+file+'.tiff',new_video, bigsize=True)

In [46]:
#IT OUTPUTS THE SAME AS ABOVE BUT WITH COLORS

# n_frames=100
# new_video=video[0:n_frames].copy()
#os.remove('second_der.csv')

#choose video or new_video to plot angles on top of the raw data or binary-centerline

#file_path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/dataset_20200605/2020-06-05_16-46-41_worm1-channel-0-/2020-06-05_16-46-41_worm1-channel-0-_MMStack_1.ome.tif'

output_path='/groups/zimmer/Ulises/code/skeleton_outputs/'

video=tiff.imread('/groups/zimmer/Ulises/code/skeleton_outputs/new/img.tiff')
#retval, mats=cv2.imreadmulti(file_path)
print(video.shape)

   
newest_video=video.copy()

new_c_video=np.full((newest_video.shape[0],newest_video.shape[1],newest_video.shape[2],3), 0, dtype='uint8')
new_c_video[:,:,:,0]=newest_video
new_c_video[:,:,:,1]=newest_video
new_c_video[:,:,:,2]=newest_video

print(new_c_video.shape)
csv_filepathK='/groups/zimmer/Ulises/code/skeleton_outputs/new3/csv_writerK.csv'
all_K=np.genfromtxt(csv_filepathK, delimiter=',')
Kmax=all_K.max()
print(all_K.shape)

csv_filepathX='/groups/zimmer/Ulises/code/skeleton_outputs/new3/csv_writerX.csv'
all_x_new=np.genfromtxt(csv_filepathX, delimiter=',')
csv_filepathY='/groups/zimmer/Ulises/code/skeleton_outputs/new3/csv_writerY.csv'
all_y_new=np.genfromtxt(csv_filepathY, delimiter=',')


#tiff.imsave('input_tiff.tiff',new_video)

# #for every frame (img) k
for img_i, img in enumerate(new_c_video):
    #for every knot
    print(img_i)
    img[:]=np.bitwise_not(img)
    for k_i, K_value in enumerate(all_K[img_i]):
        #print('K value in '+str(img_i)+ ' is: '+str(K_value))
        #print(int(K_value))
        
        #print(k_i)
        y = int(all_y_new[img_i][k_i])
        x = int(all_x_new[img_i][k_i])
        #normalize k value to 255, important to do it
        K_value=K_value/0.03*255
        if K_value>0:
            #img[x][y][0]=K_value/Kmax*2550
            #print('before norm'+K_value)
            #print('after'+K_value/0.05*255)            
            cv2.circle(img,(y,x), 3, (K_value,0,0),-1)
        if K_value<0:
            #print('before norm'+K_value)
            #print('after'+-K_value/0.05*255)  
            #img[x][y][2]=-K_value/Kmax*2550
            cv2.circle(img,(y,x), 3, (0,0,-K_value),-1)
    if img_i>1: break
tiff.imsave(output_path+'bw_colored_output5_4.tiff',new_c_video)
#tiff.imsave('new_img.tiff',new_img)

(6817, 610, 608)
(6817, 610, 608, 3)
(5900, 100)
0
1
2


In [38]:
all_K=np.genfromtxt(csv_filepath, delimiter=',')
print(all_K.shape)
print(all_K[1].shape)


csv_filepath='/groups/zimmer/Ulises/code/skeleton_outputs/new3/csv_writerX.csv'
all_x_new=np.genfromtxt(csv_filepath, delimiter=',')
print(all_K.shape)
print(all_K[1].shape)

# for k_i, K_value in enumerate(all_K[i]):
#     print('K value in '+str(k_i)+ ' is: '+str(K_value[k_i]))
#     print(K_value.shape)
#     print('\n')
#     print()
#     if k_i==3: break

(5650, 100)
(100,)


In [ ]:
#writing in an avi file
file_path='/groups/zimmer/Ulises/code/deeplabcut_projects/HeadTail-Ulises-2020-08-10/videos/2020-06-05_16-46-41_worm1-channel-0-bigtiff.avi'
#video reader
video_cap = cv2.VideoCapture(file_path)
#video writer
fps=167
frame_width = int(video_cap.get(3))
frame_height = int(video_cap.get(4))
output_filepath='/groups/zimmer/Ulises/code/deeplabcut_projects/outpy7.avi'
video_out = cv2.VideoWriter(output_filepath,cv2.VideoWriter_fourcc('M','J','P','G'), fps, (frame_width,frame_height))

#alpha parameter
alpha=.25
#counter
k=0
while(video_cap.isOpened()):
    ret,frame=video_cap.read()
    if ret == True:
        #copy for the alpha merging
        output=frame.copy()
        cv2.circle(frame, (int(head_x[k]), int(head_y[k])), 2, (20,240,20), 2)
        cv2.circle(frame, (int(tail_x[k]), int(tail_y[k])), 2, (255,20,255), 2)
        #merge to do alpha
        cv2.addWeighted(frame, alpha, output, 1-alpha, 0, output)
        #draw a black dot on the head/tail
        cv2.rectangle(output,(int(head_x[k]), int(head_y[k])),(int(head_x[k]), int(head_y[k])), (0,0,0))
        cv2.rectangle(output,(int(tail_x[k]), int(tail_y[k])),(int(tail_x[k]), int(tail_y[k])), (0,0,0))
        video_out.write(output)
        k=k+1
    #if k>20:
video_cap.release()
#video_cap.release()
video_out.release()
cv2.destroyAllWindows()

## Draw Kymographs
I want to have in python the code that I have in MATLAB.

Before running code:

Load the csv file in matlab with double click. Import.
Load paper_colormap.mat

Code:

%%
%%
%%
K=table2array(splineK);
%%Transpose
%
K=K';

%% to convert to real units (um)


K_inv=1./K;
K_px=K_inv*2.4;%2.4 is the pixel size of the BH data
K_um=1./K_px;


%%
figure();imagesc(K_um(:,1:6001))
colormap(paper_colormap)
ylabel('Segment')
xlabel('Time (s)')
c=colorbar
c.Label.String = 'Curvature  (\mum^{-1})';
caxis([-0.01 0.01])
xticks(linspace(0, 6000, 6))
xticklabels(linspace(0, 6000, 6)/200)
set(gca,'FontSize',18)